# 🏁 Proyecto: detector de discurso de odio
*Curso de NLP · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

Vas a resolver **el mismo problema de siete maneras** y compararlas: dos clásicas (BOW y TF-IDF), una supervisada
moderna (RoBERTa con ajuste fino) y cuatro con modelos de lenguaje sin entrenar (*zero-shot* y *few-shot*).

**Los datos:** el conjunto de **Davidson et al. (2017)**, *Automated Hate Speech Detection and the Problem of
Offensive Language*: tweets en inglés recogidos con el léxico de Hatebase.org y clasificados a mano por varias
personas (CrowdFlower) en **0 · discurso de odio**, **1 · lenguaje ofensivo** y **2 · ninguno**. Se carga desde
Hugging Face (`tdavidson/hate_speech_offensive`) y lo dividimos en 80 % para entrenar y 20 % para probar.

> ⚠ **Contenido sensible:** son tweets reales con insultos, groserías y discurso de odio. Se usan para enseñar a
> detectarlos.

**Entrega:** tu copia de este cuaderno en Drive (**Archivo → Guardar una copia en Drive**), con todas las celdas
ejecutadas y tus respuestas. Cada parte indica su puntaje (100 en total).

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")

In [ ]:
from pathlib import Path
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/MiAtlas-NLP")
else:
    CARPETA = Path("MiAtlas-NLP")
CARPETA.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", CARPETA)

In [ ]:
%pip install -q transformers datasets evaluate accelerate

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd

crudo = load_dataset("tdavidson/hate_speech_offensive")["train"].to_pandas().rename(columns={"class": "label"})
df_train, df_test = train_test_split(crudo[["tweet", "label"]], test_size=0.2, stratify=crudo["label"], random_state=0)
if PRUEBA_RAPIDA:
    df_train, df_test = df_train.sample(3000, random_state=0), df_test.sample(800, random_state=0)
df_train, df_test = df_train.reset_index(drop=True), df_test.reset_index(drop=True)
ETIQUETAS = {0: "discurso de odio", 1: "lenguaje ofensivo", 2: "ninguno"}
print(len(df_train), "tweets para entrenar y", len(df_test), "para probar")

In [ ]:
def matriz_confusion(y_real, y_pred, titulo="Matriz de confusión"):
    from sklearn.metrics import ConfusionMatrixDisplay
    import matplotlib.pyplot as plt
    ConfusionMatrixDisplay.from_predictions(y_real, y_pred, cmap="Blues", colorbar=False)
    plt.title(titulo)
    plt.show()

## Parte 1 · Conocer los datos (10 puntos)

- Grafica cuántos tweets hay por clase. ¿Está equilibrado? ¿Qué consecuencia tiene para entrenar y para medir?
- Calcula el largo (en palabras) por clase. ¿Hay un patrón?
- Muestra las 40 palabras más frecuentes por clase y coméntalas.

In [ ]:
# ← tu código

*Tu respuesta:*

## Parte 2 · Preprocesamiento (5 puntos)

Limpia el texto (por ejemplo: entidades HTML como `&amp;`, enlaces, menciones `@usuario`, `#`, mayúsculas) y
**justifica** cada decisión. ¿Conviene el mismo preprocesamiento para RoBERTa? ¿Por qué?

In [ ]:
# ← tu código: crea df_train["tweet_limpio"] y df_test["tweet_limpio"]

*Tu respuesta:*

## Parte 3 · BOW + un clasificador (10 puntos)

Construye la matriz de bolsa de palabras, informa el **tamaño del vocabulario** y entrena un clasificador. Reporta
precisión, *recall* y F1, en general y por clase (`classification_report`), y la matriz de confusión. Comenta.

In [ ]:
# ← tu código

## Parte 4 · TF-IDF + un clasificador (10 puntos)

Lo mismo con TF-IDF. ¿Mejora? ¿Qué pasa con la clase minoritaria (discurso de odio)? Prueba `class_weight="balanced"`.

In [ ]:
# ← tu código

## Parte 5 · RoBERTa con ajuste fino (20 puntos, con GPU)

Ajusta `FacebookAI/roberta-base` por 3 épocas (guíate por el cuaderno «🚀 Ajuste fino de RoBERTa»). Usa F1 como
métrica para elegir el mejor punto. **Guarda el modelo en Drive** para no repetir el entrenamiento (unos 20 minutos
en una GPU de Colab). Reporta las métricas y compáralas con las partes 3 y 4.

In [ ]:
# ← tu código

## Parte 6 · Un modelo de lenguaje sin ejemplos: *zero-shot* (10 puntos)

Toma una **muestra de 10 tweets por clase** del conjunto de prueba (30 en total, con una semilla fija): la usarás en
las partes 6 a 9. Diseña una instrucción que pida responder **solo 0, 1 o 2** y **justifícala**. Clasifica los 30
tweets y reporta las métricas.

In [ ]:
MODELO_LLM = "Qwen/Qwen2.5-1.5B-Instruct"   # abierto, sin trámite
# MODELO_LLM = "meta-llama/Llama-3.2-3B-Instruct"   # pide aceptar su licencia en Hugging Face y tu HF_TOKEN en Secretos

muestra = None   # ← df_test.groupby("label").sample(n=10, random_state=...)
# ← tu código

## Parte 7 · Un modelo comercial sin ejemplos (5 puntos, opcional con costo)

Usa GPT o Gemini con **la misma instrucción**. La clave va en el panel 🔑 Secretos (`OPENAI_API_KEY` o
`GEMINI_API_KEY`), nunca en el cuaderno. Clasificar 30 tweets con un modelo pequeño cuesta centavos de dólar. Si no
tienes clave, explica qué esperarías y por qué.

In [ ]:
# ← tu código (opcional)

## Parte 8 · El modelo abierto con ejemplos: *few-shot* (10 puntos)

Agrega a la instrucción **ejemplos de cada clase** tomados del conjunto de entrenamiento. Explica **con qué criterio**
los elegiste. Compara con la parte 6.

In [ ]:
# ← tu código

## Parte 9 · El modelo comercial con ejemplos (5 puntos, opcional con costo)

La misma instrucción de la parte 8, con GPT o Gemini.

In [ ]:
# ← tu código (opcional)

## Parte 10 · Los clasificadores entrenados en la misma muestra (5 puntos)

Para comparar en igualdad, clasifica los **mismos 30 tweets** con BOW, TF-IDF y RoBERTa.

In [ ]:
# ← tu código

## Parte 11 · Conclusiones (10 puntos)

- Una tabla con el F1 macro de **los siete enfoques** sobre la muestra de 30, y de los tres supervisados sobre todo
  el conjunto de prueba.
- Lo bueno y lo malo de cada enfoque: calidad, tiempo, costo, GPU, privacidad.
- ¿Cuál usarías para moderar una red social en español? ¿Qué te falta para hacerlo bien?

*Tu respuesta:*

**Fuentes:** Davidson, T., Warmsley, D., Macy, M. y Weber, I. (2017). *Automated Hate Speech Detection and the
Problem of Offensive Language*. ICWSM · [tdavidson/hate_speech_offensive](https://huggingface.co/datasets/tdavidson/hate_speech_offensive) · [RoBERTa-base](https://huggingface.co/FacebookAI/roberta-base) · [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) · [Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) · [scikit-learn · métricas](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-report)